# AbRank v2 Quickstart: Exploring 3-State Antibody–Antigen & Nanobody Affinity Ranking

This starter notebook walks through **AbRank v2** (`aurlienplissier/abrank`), a sequence- and lineage-disjoint (`<50%` antigen identity, `<80%` antibody clonal-lineage identity) 3-state benchmark for antibody–antigen and single-domain nanobody (`VHH`) binding prediction.

**What you will learn in this notebook:**
1. **Load & Audit the 4 Master Assay Tables** (`KD`, `VHH`, `Cross-Modal`, `Transfer`) and verify sequence hygiene & missing values.
2. **Explore Affinity Scales & Target Diversity** (`Viral` vs. `Non-Viral` across `192` antigen groups and `24` sources).
3. **Inspect the Zero-Leakage 3-State Splits (`{-1: Depleted, 0: Neutral, +1: Improved}`)** and explicitly verify `0%` antigen-cluster and clonal-lineage overlap.
4. **Run the Official Orientation-Invariant 3-State Evaluator** (`Directional Sign AUC`, `Hot-Spot Impact AUC`, `3-Class Ordinal AUC`, `Selective Hit AUC`, with 95% bootstrap CIs).
5. **Train & Compare Strict Zero-Leakage Sequence Baselines** on held-out `<50%`-identity antigens (`unseen_ag/test.tsv`) and export an official submission TSV compatible with `python -m abrank.evaluate`.

**Companion Repositories:**
- **Benchmark Code, Curation & Evaluator CLI:** [github.com/AI-SysBio/AbRank](https://github.com/AI-SysBio/AbRank)
- **Dual-BioFM `AbLang2-Paired + ESM-2` LogiCA Implementation:** [github.com/AI-SysBio/logica/tree/main/antibodies-abrank](https://github.com/AI-SysBio/logica/tree/main/antibodies-abrank)

## 1. Locate Dataset & Audit the 4 Master Assay Tables

We automatically locate `aurlienplissier/abrank` whether running inside a Kaggle Notebook (`/kaggle/input/abrank` or `/kaggle/input/datasets/aurlienplissier/abrank`) or locally via `kagglehub`, then inspect row counts, missingness, and canonical target coverage.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

def find_abrank_root() -> Path:
    candidates = [
        Path("/kaggle/input/abrank"),
        Path("/kaggle/input/datasets/aurlienplissier/abrank"),
        Path("data"),
    ]
    for c in candidates:
        if (c / "benchmark" / "assays_kd.tsv").exists():
            return c
        if (c / "assays_kd.tsv").exists():
            return c.parent
    # Fallback for local/Colab execution via kagglehub
    import kagglehub
    return Path(kagglehub.dataset_download("aurlienplissier/abrank"))

ROOT = find_abrank_root()
BDIR = ROOT / "benchmark" if (ROOT / "benchmark").is_dir() else ROOT
MDIR = ROOT / "metadata"
print(f"Resolved AbRank v2 root : {ROOT}")
print(f"Benchmark directory     : {BDIR}")

MASTER_SPECS = [
    ("Core Paired KD", "assays_kd.tsv"),
    ("Single-Domain VHH KD", "assays_vhh.tsv"),
    ("Cross-Modal (Enrich/ELISA/DDG/EC50/DMS)", "assays_cross_modal.tsv"),
    ("Transfer (Viral IC50 & RBD Escape)", "assays_transfer.tsv"),
]

inventory_rows = []
tables = {}
for label, fname in MASTER_SPECS:
    df = pd.read_csv(BDIR / fname, sep="\t", low_memory=False)
    tables[fname] = df
    n_miss_score = int(df["binding_score"].isna().sum())
    n_miss_vh = int(df["Ab_heavy_chain_seq"].isna().sum())
    n_miss_ag = int(df["Ag_seq"].isna().sum())
    inventory_rows.append({
        "Track": label,
        "File": fname,
        "Assays": len(df),
        "Unique AbID": df["AbID"].nunique(),
        "Ag Groups": df["Ag_group"].nunique(),
        "Sources": df["Dataset"].nunique(),
        "Modalities": ", ".join(sorted(df["Assay_modality"].astype(str).unique())),
        "Missing Score/VH/Ag": f"{n_miss_score}/{n_miss_vh}/{n_miss_ag}",
    })

inv_df = pd.DataFrame(inventory_rows)
display(inv_df)

kd_df = tables["assays_kd.tsv"]
print("\nSample rows from benchmark/assays_kd.tsv:")
display(kd_df[["AbAgID", "Dataset", "Ag_group", "Ag_type", "Ag_cluster", "Ab_lineage", "binding_score", "WT_PDB"]].head(5))

**Data Hygiene Check:** All four master tables contain **zero missing values** in `binding_score`, `Ab_heavy_chain_seq`, and `Ag_seq` (`0/0/0`). Note that `Ab_light_chain_seq` is intentionally empty for single-domain `VHH` nanobodies in `assays_vhh.tsv` (which lack a light chain), and `WT_PDB` is populated whenever an experimental co-crystal exists (`340` solved co-crystals out of `851` wild-type parental scaffolds in `metadata/cofolding_manifest.tsv`).

---

## 2. Visualize Affinity Distributions (`pKd`) & Viral vs. Non-Viral Target Coverage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2), layout="constrained")

# Panel A: pKd distribution across top curated KD sources (>= 40 assays)
ds_counts = kd_df["Dataset"].value_counts()
top_ds = ds_counts[ds_counts >= 40].index.tolist()
box_data = [kd_df.loc[kd_df["Dataset"] == d, "binding_score"].dropna().to_numpy() for d in top_ds]
colors_a = plt.cm.tab20(np.linspace(0, 0.85, len(top_ds)))

bp = axes[0].boxplot(box_data, patch_artist=True, showfliers=False, widths=0.62)
for patch, col in zip(bp["boxes"], colors_a):
    patch.set_facecolor(col)
    patch.set_alpha(0.82)
for med in bp["medians"]:
    med.set_color("black")
    med.set_linewidth(1.4)

axes[0].set_xticks(range(1, len(top_ds) + 1))
axes[0].set_xticklabels(top_ds, rotation=38, ha="right")
axes[0].set_ylabel("Standardized binding_score (pKd = 9 - log10 Kd [nM])")
axes[0].set_title("A. Core Paired-Chain pKd Distribution by Source Study")
axes[0].grid(axis="y", alpha=0.25)

# Panel B: Viral vs. Non-Viral unique antigen groups across tracks
track_names = ["All Unique", "Core KD", "VHH KD", "Cross-Modal", "Transfer"]
track_dfs = [
    pd.concat([tables[f][["Ag_group", "Ag_type"]] for _, f in MASTER_SPECS], ignore_index=True),
    tables["assays_kd.tsv"][["Ag_group", "Ag_type"]],
    tables["assays_vhh.tsv"][["Ag_group", "Ag_type"]],
    tables["assays_cross_modal.tsv"][["Ag_group", "Ag_type"]],
    tables["assays_transfer.tsv"][["Ag_group", "Ag_type"]],
]
non_viral_cnts, viral_cnts = [], []
for tdf in track_dfs:
    dedup = tdf.drop_duplicates("Ag_group")
    non_viral_cnts.append(int((dedup["Ag_type"] == "Non-Viral").sum()))
    viral_cnts.append(int((dedup["Ag_type"] == "Viral").sum()))

y_pos = np.arange(len(track_names))
axes[1].barh(y_pos, non_viral_cnts, color="#2b5c8f", label="Non-Viral")
axes[1].barh(y_pos, viral_cnts, left=non_viral_cnts, color="#e07b39", label="Viral")
for i, (nv, v) in enumerate(zip(non_viral_cnts, viral_cnts)):
    axes[1].text(nv + v + 3, i, f"{nv + v}", va="center", fontweight="bold")

axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(track_names)
axes[1].invert_yaxis()
axes[1].set_xlabel("Unique Canonical Antigen Groups")
axes[1].set_title("B. Antigen Diversity Across AbRank v2 Tracks")
axes[1].set_xlim(0, max(nv + v for nv, v in zip(non_viral_cnts, viral_cnts)) * 1.18)
axes[1].legend(frameon=False, loc="lower right")
axes[1].grid(axis="x", alpha=0.25)

plt.show()

**Key EDA Observations:**
- **Dynamic Range & Cross-Lab Offsets:** Core paired-chain `pKd` spans nanomolar to sub-picomolar binders (`pKd ~ 5–11`), with noticeable median shifts between individual experimental platforms (e.g., `Origin1` vs. `AIntibody` vs. `SAbDab`). Evaluating **within-assay relative pairs (`Δbinding_score`)** cancels uncalibrated cross-laboratory offsets.
- **Broad Non-Viral Coverage:** Unlike viral-only benchmarks, `>80%` of AbRank's `192` harmonized antigen groups are non-viral therapeutic and structural targets.

---

## 3. Zero-Leakage 3-State Splits (`{-1, 0, +1}`) & Leakage Audit

AbRank partitions homologous antigen clusters (`<50%` identity) and antibody clonal lineages (`<80%` Fv/VHH identity) **prior to pair construction**. Below we load all Core Suite splits, programmatically assert **0% leakage**, and visualize the 3-state noise buffer.

In [ ]:
SPDIR = BDIR / "splits"
SPLIT_SPECS = [
    ("unseen_ag / train", SPDIR / "unseen_ag" / "train.tsv"),
    ("unseen_ag / val", SPDIR / "unseen_ag" / "val.tsv"),
    ("unseen_ag / test", SPDIR / "unseen_ag" / "test.tsv"),
    ("unseen_ag / test_cand", SPDIR / "unseen_ag" / "test_candidates.tsv"),
    ("generative / pairs", SPDIR / "generative" / "generative_pairs.tsv"),
    ("transfer / ic50", SPDIR / "transfer" / "ic50_test.tsv"),
    ("transfer / escape", SPDIR / "transfer" / "escape_test.tsv"),
    ("cross_modal / test", SPDIR / "cross_modal" / "modality_test.tsv"),
    ("vhh / test", SPDIR / "vhh" / "vhh_ranking_test.tsv"),
]

split_dfs = {}
split_summary = []
for name, path in SPLIT_SPECS:
    sdf = pd.read_csv(path, sep="\t", low_memory=False)
    split_dfs[name] = sdf
    vc = sdf["label"].value_counts()
    split_summary.append({
        "Split": name,
        "Total Pairs": len(sdf),
        "Depleted (-1)": int(vc.get(-1, 0)),
        "Neutral (0)": int(vc.get(0, 0)),
        "Improved (+1)": int(vc.get(1, 0)),
        "+1 / (+1 + -1)": round(float(vc.get(1, 0) / max(vc.get(1, 0) + vc.get(-1, 0), 1)), 3),
    })

split_sum_df = pd.DataFrame(split_summary)
display(split_sum_df)

# Explicit zero-leakage assertion on unseen_ag train / val / test
id2ag = kd_df.drop_duplicates("AbAgID").set_index("AbAgID")["Ag_cluster"].to_dict()
id2lin = kd_df.drop_duplicates("AbAgID").set_index("AbAgID")["Ab_lineage"].to_dict()
part_ag, part_lin = {}, {}
for part in ("train", "val", "test"):
    df_p = split_dfs[f"unseen_ag / {part}"]
    ids = set(df_p["AbAgID_1"]).union(df_p["AbAgID_2"])
    part_ag[part] = {id2ag[i] for i in ids if i in id2ag}
    part_lin[part] = {id2lin[i] for i in ids if i in id2lin}

assert len(part_ag["train"] & part_ag["val"]) == 0
assert len(part_ag["train"] & part_ag["test"]) == 0
assert len(part_ag["val"] & part_ag["test"]) == 0
assert len(part_lin["train"] & part_lin["test"]) == 0
print(
    f"[VERIFIED 0% LEAKAGE] Ag clusters: train={len(part_ag['train'])}, val={len(part_ag['val'])}, "
    f"test={len(part_ag['test'])} (overlap=0) | Ab lineages train∩test=0"
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), layout="constrained")

# Panel A: Local L1-L3 ΔpKd distribution highlighting the 3-state noise buffer
tr_loc = split_dfs["unseen_ag / train"]
tr_loc = tr_loc[tr_loc["Category"].isin(["L1", "L2", "L3"])]
bins = np.linspace(-4.0, 4.0, 65)
for lbl, col, name in [(-1, "#c44e52", "Depleted (-1: Δ <= -1.0)"), (0, "#4c72b0", "Neutral (0: |Δ| <= 0.30)"), (1, "#55a868", "Improved (+1: Δ >= +1.0)")]:
    vals = tr_loc.loc[tr_loc["label"] == lbl, "binding_diff"].clip(-4.0, 4.0)
    axes[0].hist(vals, bins=bins, alpha=0.80, color=col, label=name)
for thr, ls in [(-1.0, "--"), (1.0, "--"), (-0.30, ":"), (0.30, ":")]:
    axes[0].axvline(thr, color="#333333", linestyle=ls, linewidth=1.1)
axes[0].set_xlabel("Signed Affinity Difference ΔpKd = pKd(1) - pKd(2)")
axes[0].set_ylabel("Pair Count")
axes[0].set_title("A. 3-State Noise-Buffered Distribution (Local L1–L3 Pairs)")
axes[0].legend(frameon=False, fontsize=8.5, loc="upper right")
axes[0].grid(axis="y", alpha=0.25)

# Panel B: 3-state proportion across all Core Suite splits
tot = split_sum_df["Total Pairs"].to_numpy(dtype=float)
p_neg = 100.0 * split_sum_df["Depleted (-1)"].to_numpy() / tot
p_neu = 100.0 * split_sum_df["Neutral (0)"].to_numpy() / tot
p_pos = 100.0 * split_sum_df["Improved (+1)"].to_numpy() / tot
y_s = np.arange(len(split_sum_df))

axes[1].barh(y_s, p_neg, color="#c44e52", label="Depleted (-1)")
axes[1].barh(y_s, p_neu, left=p_neg, color="#4c72b0", label="Neutral (0)")
axes[1].barh(y_s, p_pos, left=p_neg + p_neu, color="#55a868", label="Improved (+1)")
axes[1].set_yticks(y_s)
axes[1].set_yticklabels(split_sum_df["Split"])
axes[1].invert_yaxis()
axes[1].set_xlabel("Share of Pairs (%)")
axes[1].set_xlim(0, 100)
axes[1].set_title("B. Exact 50/50 Impactful Balance & Neutral Share Across Splits")
axes[1].legend(frameon=False, ncol=3, loc="lower center", bbox_to_anchor=(0.5, -0.24))
axes[1].grid(axis="x", alpha=0.25)

plt.show()

**Why 3-State Evaluation?** Every split is deterministically balanced `50/50` between `+1` and `-1` impactful pairs while retaining a calibrated `0` (neutral) noise band (`|ΔpKd| ≤ 0.30`). This prevents models from trivially exploiting orientation bias and simultaneously tests whether a model can distinguish **hot-spot edits (`±1` vs. `0`)** from **beneficial vs. deleterious mutations (`+1` vs. `-1`)**.

---

## 4. Official Orientation-Invariant 3-State Evaluator

Below is the exact reference implementation of AbRank's orientation-invariant 3-state metrics (identical to `abrank.evaluate` in [AI-SysBio/AbRank](https://github.com/AI-SysBio/AbRank)), along with unit verification of oracle (`1.0000`), constant (`0.5000`), and pair-flip invariance.

In [ ]:
def compute_3state_metrics(labels, binding_diff, pred_diff, pred_impact=None):
    """Orientation-invariant 3-state metrics (Sign, Impact, 3-Class Ordinal, Selective Hit AUC)."""
    lbl = np.asarray(labels, dtype=int)
    bdiff = np.asarray(binding_diff, dtype=np.float64)
    pdiff = np.asarray(pred_diff, dtype=np.float64)
    pimp = np.abs(pdiff) if pred_impact is None else np.asarray(pred_impact, dtype=np.float64)

    imp_mask = lbl != 0
    neu_mask = lbl == 0
    n_imp, n_neu = int(imp_mask.sum()), int(neu_mask.sum())

    if n_imp > 0:
        y_imp = (bdiff[imp_mask] > 0).astype(int)
        p_imp = pdiff[imp_mask]
        auc_dir = float(roc_auc_score(np.concatenate([y_imp, 1 - y_imp]), np.concatenate([p_imp, -p_imp])))
    else:
        auc_dir = float("nan")

    if n_imp > 0 and n_neu > 0:
        auc_impact = float(roc_auc_score(imp_mask.astype(int), pimp))
        s_plus = np.where(bdiff[imp_mask] > 0, pdiff[imp_mask], -pdiff[imp_mask])
        s_minus = -s_plus
        s_zero = np.concatenate([pdiff[neu_mask], -pdiff[neu_mask]])
        auc_0p = float(roc_auc_score(
            np.concatenate([np.ones(n_imp, dtype=int), np.zeros(2 * n_neu, dtype=int)]),
            np.concatenate([s_plus, s_zero]),
        ))
        w_pm = float(n_imp) * float(n_imp)
        w_0p = 2.0 * float(n_imp) * float(n_neu)
        auc_ordinal = (w_pm * auc_dir + 2.0 * w_0p * auc_0p) / (w_pm + 2.0 * w_0p)
        auc_hit = float(roc_auc_score(
            np.concatenate([np.ones(n_imp, dtype=int), np.zeros(n_imp + 2 * n_neu, dtype=int)]),
            np.concatenate([s_plus, s_minus, s_zero]),
        ))
    elif n_imp > 0:
        auc_impact, auc_ordinal, auc_hit = float("nan"), auc_dir, auc_dir
    else:
        auc_impact = auc_ordinal = auc_hit = float("nan")

    _r = lambda v: round(float(v), 4) if np.isfinite(v) else None
    return {
        "n_pairs": int(len(lbl)),
        "n_impactful": n_imp,
        "n_neutral": n_neu,
        "auc_dir": _r(auc_dir),
        "auc_impact": _r(auc_impact),
        "auc_ordinal": _r(auc_ordinal),
        "auc_selective_hit": _r(auc_hit),
    }


def compute_per_antigen_macro_metrics(pairs_df, pred_diff, pred_impact=None, min_pairs=5, n_bootstrap=300, seed=42):
    """Equal-weight per-antigen macro-average across same-antigen strata (>= min_pairs) with 95% CIs."""
    df = pairs_df.reset_index(drop=True)
    pdiff = np.asarray(pred_diff, dtype=np.float64)
    pimp = np.abs(pdiff) if pred_impact is None else np.asarray(pred_impact, dtype=np.float64)
    lbl = df["label"].to_numpy(dtype=int)
    bdiff = df["binding_diff"].to_numpy(dtype=np.float64)

    same_ag = (df["Category"] != "Global").to_numpy() & (df["Ag_group_1"].astype(str) == df["Ag_group_2"].astype(str)).to_numpy()
    sub_df = df.iloc[np.flatnonzero(same_ag)]
    grp_indices = [(str(ag), g.index.to_numpy()) for ag, g in sub_df.groupby("Ag_group_1", sort=True) if len(g) >= min_pairs]

    per_ag = {ag: compute_3state_metrics(lbl[ix], bdiff[ix], pdiff[ix], pimp[ix]) for ag, ix in grp_indices}
    def _mean(key):
        vals = [m[key] for m in per_ag.values() if m[key] is not None]
        return round(float(np.mean(vals)), 4) if vals else None

    out = {
        "n_antigens": len(grp_indices),
        "auc_dir": _mean("auc_dir"),
        "auc_impact": _mean("auc_impact"),
        "auc_ordinal": _mean("auc_ordinal"),
        "auc_selective_hit": _mean("auc_selective_hit"),
        "per_antigen": per_ag,
    }
    if n_bootstrap > 0 and grp_indices:
        rng = np.random.default_rng(seed)
        boots = {k: [] for k in ("auc_dir", "auc_impact", "auc_ordinal", "auc_selective_hit")}
        for _ in range(n_bootstrap):
            bm = [compute_3state_metrics(lbl[s], bdiff[s], pdiff[s], pimp[s]) for _, ix in grp_indices for s in [ix[rng.integers(0, len(ix), len(ix))]]]
            for k in boots:
                vals = [m[k] for m in bm if m[k] is not None]
                if vals:
                    boots[k].append(float(np.mean(vals)))
        for k, arr in boots.items():
            out[f"{k}_ci95"] = [round(float(np.percentile(arr, 2.5)), 4), round(float(np.percentile(arr, 97.5)), 4)] if arr else [0.5, 0.5]
    return out

# Sanity checks on unseen_ag/test.tsv
te_df = split_dfs["unseen_ag / test"]
m_oracle = compute_3state_metrics(te_df["label"], te_df["binding_diff"], te_df["binding_diff"])
m_const = compute_3state_metrics(te_df["label"], te_df["binding_diff"], np.zeros(len(te_df)))
assert m_oracle["auc_ordinal"] == 1.0 and m_oracle["auc_dir"] == 1.0 and m_oracle["auc_impact"] == 1.0
assert m_const["auc_ordinal"] == 0.5 and m_const["auc_dir"] == 0.5 and m_const["auc_impact"] == 0.5
print("Evaluator sanity checks passed: Oracle =", m_oracle["auc_ordinal"], "| Random/Zero =", m_const["auc_ordinal"])

## 5. End-to-End Zero-Leakage Sequence Baseline (`unseen_ag`)

To demonstrate a clean, leak-free modeling workflow, we compare three baselines on held-out `<50%`-identity unseen antigens (`unseen_ag/test.tsv`):
1. **Naive Constant Baseline** (chance = `0.5000`)
2. **Antisymmetric Logistic Regression (`Biophys-LogReg`)**: Uses strictly odd biophysical difference features $\phi(1) - \phi(2)$ with `fit_intercept=False` (guaranteeing exact odd antisymmetry $f(A, B) = -f(B, A)$) paired with an even-symmetric $|\phi(1) - \phi(2)|$ impact classifier.
3. **Antisymmetric Margin Classifier (`Biophys-Ridge`)**: Regularized $L_2$ margin ranker tuned strictly on `unseen_ag/val.tsv`.

All preprocessing scalers (`StandardScaler`) and hyperparameters are fit **exclusively on `train.tsv`** and selected on **`val.tsv`** before touching **`test.tsv`**.

In [ ]:
# Lightweight physicochemical sequence descriptors per chain (VH, VL, Ag)
KD_HYDRO = {"A":1.8,"C":2.5,"D":-3.5,"E":-3.5,"F":2.8,"G":-0.4,"H":-3.2,"I":4.5,"K":-3.9,"L":3.8,
            "M":1.9,"N":-3.5,"P":-1.6,"Q":-3.5,"R":-4.5,"S":-0.8,"T":-0.7,"V":4.2,"W":-0.9,"Y":-1.3}
CHARGE = {"D":-1.0, "E":-1.0, "K":1.0, "R":1.0, "H":0.25}
AROM = set("FWY")
POLAR = set("STNQCY")
AA_LIST = "ACDEFGHIKLMNPQRSTVWY"

def seq_biophys_vec(seq: str) -> np.ndarray:
    s = str(seq).strip().upper()
    L = max(len(s), 1)
    hydro = sum(KD_HYDRO.get(c, 0.0) for c in s) / L
    chg = sum(CHARGE.get(c, 0.0) for c in s) / L
    arom = sum(c in AROM for c in s) / L
    polar = sum(c in POLAR for c in s) / L
    aa_freq = [s.count(a) / L for a in AA_LIST]
    return np.array([np.log1p(L), hydro, chg, arom, polar] + aa_freq, dtype=np.float32)

tr_df = split_dfs["unseen_ag / train"]
va_df = split_dfs["unseen_ag / val"]
te_df = split_dfs["unseen_ag / test"]

# Featurize only unique complexes referenced across train/val/test
needed_ids = set().union(*[set(d["AbAgID_1"]).union(d["AbAgID_2"]) for d in (tr_df, va_df, te_df)])
sub_kd = kd_df[kd_df["AbAgID"].isin(needed_ids)].drop_duplicates("AbAgID")
seq_cache = {}
for r in sub_kd.itertuples(index=False):
    seq_cache[r.AbAgID] = np.concatenate([
        seq_biophys_vec(r.Ab_heavy_chain_seq),
        seq_biophys_vec(r.Ab_light_chain_seq),
        seq_biophys_vec(r.Ag_seq),
    ])

def build_pair_matrices(df):
    v1 = np.vstack([seq_cache[i] for i in df["AbAgID_1"]])
    v2 = np.vstack([seq_cache[i] for i in df["AbAgID_2"]])
    x_odd = v1 - v2                     # Exact odd antisymmetry: f(1,2) = -f(2,1)
    x_even = np.abs(v1 - v2)            # Exact even symmetry:    q(1,2) = +q(2,1)
    return x_odd, x_even

X_tr_odd, X_tr_even = build_pair_matrices(tr_df)
X_va_odd, X_va_even = build_pair_matrices(va_df)
X_te_odd, X_te_even = build_pair_matrices(te_df)

# Fit scalers STRICTLY on training data (with_mean=False preserves odd antisymmetry)
sc_odd = StandardScaler(with_mean=False).fit(X_tr_odd)
sc_even = StandardScaler().fit(X_tr_even)

Z_tr_o, Z_va_o, Z_te_o = sc_odd.transform(X_tr_odd), sc_odd.transform(X_va_odd), sc_odd.transform(X_te_odd)
Z_tr_e, Z_va_e, Z_te_e = sc_even.transform(X_tr_even), sc_even.transform(X_va_even), sc_even.transform(X_te_even)

# Train directional models on impactful pairs (label != 0) and impact detector on all local pairs
imp_tr = tr_df["label"].to_numpy() != 0
y_dir_tr = (tr_df.loc[imp_tr, "label"].to_numpy() > 0).astype(int)
loc_tr_m = tr_df["Category"].isin(["L1", "L2", "L3"]).to_numpy()
y_imp_tr = (tr_df.loc[loc_tr_m, "label"].to_numpy() != 0).astype(int)

# Tune impact classifier on validation
best_imp_clf, best_imp_auc = None, -1.0
for C in (1e-3, 1e-2, 1e-1):
    clf_i = LogisticRegression(C=C, class_weight="balanced", max_iter=500).fit(Z_tr_e[loc_tr_m], y_imp_tr)
    p_v = clf_i.predict_proba(Z_va_e)[:, 1]
    sc_v = compute_per_antigen_macro_metrics(va_df, np.zeros(len(va_df)), p_v, n_bootstrap=0)["auc_impact"] or 0.5
    if sc_v > best_imp_auc:
        best_imp_auc, best_imp_clf = sc_v, clf_i

p_imp_te = best_imp_clf.predict_proba(Z_te_e)[:, 1]

# Model 1: Antisymmetric Logistic Regression (fit_intercept=False)
best_lr, best_lr_sc = None, -1.0
for C in (1e-4, 1e-3, 1e-2, 1e-1):
    lr = LogisticRegression(C=C, fit_intercept=False, max_iter=500).fit(Z_tr_o[imp_tr], y_dir_tr)
    dv = lr.decision_function(Z_va_o)
    sc_v = compute_per_antigen_macro_metrics(va_df, dv, best_imp_clf.predict_proba(Z_va_e)[:, 1], n_bootstrap=0)["auc_ordinal"] or 0.5
    if sc_v > best_lr_sc:
        best_lr_sc, best_lr = sc_v, lr

# Model 2: Antisymmetric Ridge Classifier (fit_intercept=False)
best_rc, best_rc_sc = None, -1.0
for alpha in (1e2, 1e3, 1e4, 1e5):
    rc = RidgeClassifier(alpha=alpha, fit_intercept=False).fit(Z_tr_o[imp_tr], y_dir_tr)
    dv = rc.decision_function(Z_va_o)
    sc_v = compute_per_antigen_macro_metrics(va_df, dv, best_imp_clf.predict_proba(Z_va_e)[:, 1], n_bootstrap=0)["auc_ordinal"] or 0.5
    if sc_v > best_rc_sc:
        best_rc_sc, best_rc = sc_v, rc

pred_lr_d = p_imp_te * np.tanh(best_lr.decision_function(Z_te_o))
pred_rc_d = p_imp_te * np.tanh(best_rc.decision_function(Z_te_o))

# Verify exact odd antisymmetry: f(B, A) == -f(A, B)
assert np.allclose(best_lr.decision_function(-Z_te_o), -best_lr.decision_function(Z_te_o), atol=1e-6)

eval_results = {
    "Naive Zero Baseline": compute_per_antigen_macro_metrics(te_df, np.zeros(len(te_df)), np.zeros(len(te_df)), n_bootstrap=300),
    "Biophys-LogReg (3-State)": compute_per_antigen_macro_metrics(te_df, pred_lr_d, p_imp_te, n_bootstrap=300),
    "Biophys-Ridge (3-State)": compute_per_antigen_macro_metrics(te_df, pred_rc_d, p_imp_te, n_bootstrap=300),
}

comp_rows = []
for mname, res in eval_results.items():
    comp_rows.append({
        "Model": mname,
        "Antigens": res["n_antigens"],
        "Sign AUC (95% CI)": f"{res['auc_dir']:.4f} {res['auc_dir_ci95']}",
        "Impact AUC (95% CI)": f"{res['auc_impact']:.4f} {res['auc_impact_ci95']}",
        "Ordinal AUC (95% CI)": f"{res['auc_ordinal']:.4f} {res['auc_ordinal_ci95']}",
        "Selective Hit AUC": res["auc_selective_hit"],
    })
display(pd.DataFrame(comp_rows))

# Export official AbRank submission TSV format (`pair_id`, `pred_diff`, `pred_impact`)
out_path = Path("predictions_unseen_ag_test.tsv")
pd.DataFrame({
    "pair_id": te_df["pair_id"],
    "pred_diff": pred_lr_d,
    "pred_impact": p_imp_te,
}).to_csv(out_path, sep="\t", index=False)
print(f"Saved official AbRank submission file -> {out_path.resolve()} ({len(te_df)} rows)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), layout="constrained")

# Panel A: Model comparison with 95% bootstrap CIs on unseen_ag/test.tsv
metrics_keys = [("auc_dir", "Sign AUC", "#4c72b0"), ("auc_impact", "Impact AUC", "#dd8452"), ("auc_ordinal", "3-Class Ordinal AUC", "#55a868")]
model_names = list(eval_results.keys())
x = np.arange(len(model_names))
w = 0.25
for j, (mk, mlabel, col) in enumerate(metrics_keys):
    vals = [eval_results[m][mk] for m in model_names]
    lo = [max(0.0, eval_results[m][mk] - eval_results[m][f"{mk}_ci95"][0]) for m in model_names]
    hi = [max(0.0, eval_results[m][f"{mk}_ci95"][1] - eval_results[m][mk]) for m in model_names]
    axes[0].bar(x + (j - 1) * w, vals, width=w, color=col, yerr=[lo, hi], capsize=3, label=mlabel)
axes[0].axhline(0.5, color="#666666", linestyle="--", linewidth=1.0)
axes[0].set_xticks(x)
axes[0].set_xticklabels(model_names, rotation=12, ha="right")
axes[0].set_ylim(0.35, 0.85)
axes[0].set_ylabel("Per-Antigen Macro ROC-AUC")
axes[0].set_title("A. Unseen-Antigen Test Performance (95% Bootstrap CI)")
axes[0].legend(frameon=False, loc="upper left")
axes[0].grid(axis="y", alpha=0.25)

# Panel B: Per-antigen slice breakdown for Biophys-LogReg
per_ag = eval_results["Biophys-LogReg (3-State)"]["per_antigen"]
ag_names = list(per_ag.keys())
ag_ord = [per_ag[a]["auc_ordinal"] if per_ag[a]["auc_ordinal"] is not None else 0.5 for a in ag_names]
ag_imp = [per_ag[a]["auc_impact"] if per_ag[a]["auc_impact"] is not None else 0.5 for a in ag_names]
ya = np.arange(len(ag_names))
axes[1].barh(ya - 0.18, ag_ord, height=0.34, color="#55a868", label="3-Class Ordinal AUC")
axes[1].barh(ya + 0.18, ag_imp, height=0.34, color="#dd8452", label="Impact AUC")
axes[1].axvline(0.5, color="#666666", linestyle="--", linewidth=1.0)
axes[1].set_yticks(ya)
axes[1].set_yticklabels(ag_names)
axes[1].invert_yaxis()
axes[1].set_xlim(0.25, 0.95)
axes[1].set_xlabel("ROC-AUC")
axes[1].set_title("B. Per-Antigen Generalization Slices (Held-Out <50% Identity)")
axes[1].legend(frameon=False, loc="lower right")
axes[1].grid(axis="x", alpha=0.25)

plt.show()

## Summary

### Data Analysis Key Findings
- **Complete, Standardized Multi-Modality Coverage:** AbRank v2 aggregates `120,721` deduplicated paired-chain `KD` assays (`148` antigen groups, `16` sources), `298` single-domain `VHH` nanobody `KD` assays (`50` antigen groups), `79,879` cross-modal assays (`Enrichment`, `ELISA`, `DDG`, `EC50`, `DMS`), and `229,738` functional viral `IC50` / `RBD Escape` assays with `0` missing scores or heavy/antigen sequences.
- **Verified Zero-Leakage & Balanced 3-State Splits:** Explicit cluster-level verification confirms `0` shared `<50%`-identity antigen clusters and `0` shared `<80%`-identity clonal lineages across `unseen_ag/{train,val,test}.tsv`, with exact `50/50` positive/negative balance among impactful pairs alongside the empirically calibrated `|ΔpKd| ≤ 0.30` neutral noise band.
- **Hardness of Unseen-Antigen Generalization:** Simple sequence-composition baselines (`Biophys-LogReg`, `Biophys-Ridge`) establish a fast, strictly antisymmetric starting point on held-out antigens while highlighting substantial headroom for dual-PLM cross-attention (`AbLang2-Paired + ESM-2` LogiCA) and WT-3D structure-conditioned models (`ProteinMPNN`, `ANTIPASTI`, `GearBind`).

### Insights or Next Steps
- **Score on all 4 Core Suites via the Official CLI:** Install [AI-SysBio/AbRank](https://github.com/AI-SysBio/AbRank) (`pip install git+https://github.com/AI-SysBio/AbRank.git`) and pass any `pair_id,pred_diff[,pred_impact]` or pointwise `AbAgID,pred_score` TSV directly to `python -m abrank.evaluate --suite {maturation,discovery,generative,transfer} --kaggle-dataset aurlienplissier/abrank --preds <file.tsv>`.
- **Upgrade to Dual-BioFM Native-Logit Cross-Attention:** Run the ready-to-use `AbLang2-Paired + ESM-2` LogiCA training, evaluation, paratope dependency map, and CDR-H3 Gibbs affinity maturation scripts in [AI-SysBio/logica/antibodies-abrank](https://github.com/AI-SysBio/logica/tree/main/antibodies-abrank) (pointing directly to `--kaggle-dataset aurlienplissier/abrank`).